# prompting_check — consolidated results for the prompting arm
Reads the frozen splits and `scores/` only; generates nothing. The main score-derived arm-A result tables and diagnostics are reproduced here from the frozen score files; no generation or judging is rerun.

Arm A has three levers. Instruction design and negative-prompt calibration are measured on the 302-row screen slice against a screen baseline generated in the same session. Reference conditioning is measured on the 488-row test split against the base model with references removed. The noise floor for both is measured, not assumed: an identical configuration regenerated cross-session, and the judge re-scoring the same image.

## 1. Load and integrity

In [1]:
from google.colab import drive
import os, json, math, re
from collections import defaultdict
import numpy as np
from scipy import stats
drive.mount("/content/drive")

DRIVE  = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
SCORES = f"{DRIVE}/scores"

def load_by_uid(path):
    d = {}
    for l in open(path, encoding="utf-8"):
        r = json.loads(l)
        if "error" in r: continue
        d[r["uid"]] = r
    return d

ov  = lambda r: math.sqrt(max(r["PF"], 0) * max(r["SC"], 0))
MET = [("PF", lambda r: r["PF"]), ("SC", lambda r: r["SC"]), ("Overall", ov)]

def guessed(path):
    n = 0
    for l in open(path, encoding="utf-8"):
        r = json.loads(l)
        if str(r.get("PF_reason", "")).startswith("guess_if_cannot_parse") \
           or str(r.get("SC_reason", "")).startswith("guess_if_cannot_parse"): n += 1
    return n

L1 = ("a1_complete", "a2_identity", "a3_both", "a4_style", "a5_filler")     # instruction rubrics
L2 = ("n_contrastive", "n_contrastive2", "n_filler", "n_empty")            # negative prompt

# 488-row test split
BASE  = load_by_uid(f"{SCORES}/test__baseline.jsonl")
ZERO  = load_by_uid(f"{SCORES}/test__zero_ref.jsonl")        # base model, references removed
# 302-row screen slice, carved from train; never overlaps test or val
SB    = load_by_uid(f"{SCORES}/screen__baseline.jsonl")
SBR   = load_by_uid(f"{SCORES}/screen__baseline__rerun.jsonl")   # identical config, later session
ARM   = {a: load_by_uid(f"{SCORES}/screen__{a}.jsonl") for a in L1 + L2}

META = {r["output_image"] + "|" + "|".join(sorted(r["input_images"])): r
        for r in [json.loads(l) for l in open(f"{DRIVE}/test.jsonl", encoding="utf-8")]}

I  = sorted(set(BASE) & set(ZERO))
S  = sorted(set(SB) & set(SBR) & set.intersection(*[set(d) for d in ARM.values()]))

files = ["test__baseline", "test__zero_ref", "screen__baseline", "screen__baseline__rerun"] \
        + [f"screen__{a}" for a in L1 + L2]
print(f"test rows paired across baseline and zero-reference : {len(I)}")
print(f"screen rows paired across baseline, rerun and all nine lever arms : {len(S)}")
print(f"screen targets represented : {len({k.split('|')[0] for k in S})}")
print(f"guessed judge scores across every file above : {sum(guessed(f'{SCORES}/{f}.jsonl') for f in files)}")

Mounted at /content/drive
test rows paired across baseline and zero-reference : 488
screen rows paired across baseline, rerun and all nine lever arms : 302
screen targets represented : 62
guessed judge scores across every file above : 0


## 2. Headline — the prompting baseline on the test split

In [2]:
def paired(A, B, idx, f):
    d = np.array([f(A[k]) - f(B[k]) for k in idx], float)
    n = len(d); se = d.std(ddof=1) / math.sqrt(n)
    p = stats.ttest_rel([f(A[k]) for k in idx], [f(B[k]) for k in idx])[1]
    tc = stats.t.ppf(0.95, n - 1)
    return dict(n=n, mean=d.mean(), sd=d.std(ddof=1), se=se, p=p,
                lo=d.mean() - tc * se, hi=d.mean() + tc * se, d=d)

def show(lab, A, B, idx, mets=MET):
    for name, f in mets:
        s = paired(A, B, idx, f)
        print(f"  {lab:32s} {name:8s} n={s['n']:4d}  delta {s['mean']:+7.3f}  sigma_d {s['sd']:5.3f}  "
              f"SE {s['se']:.3f}  p {s['p']:.3g}  90%CI [{s['lo']:+.3f}, {s['hi']:+.3f}]")

BOUND = 0.15
def tost(A, B, idx, bound=BOUND):
    s = paired(A, B, idx, ov); n = s["n"]
    p_lo = 1 - stats.t.cdf((s["mean"] + bound) / s["se"], n - 1)
    p_hi = stats.t.cdf((s["mean"] - bound) / s["se"], n - 1)
    return s, p_lo, p_hi, (p_lo < 0.05 and p_hi < 0.05)

def delta_icc(A, B, idx):
    g = defaultdict(list)
    for k in idx: g[k.split("|")[0]].append(ov(A[k]) - ov(B[k]))
    allv = np.concatenate([np.array(v) for v in g.values()]); kbar = np.mean([len(v) for v in g.values()])
    grand = allv.mean()
    msb = sum(len(v) * (np.mean(v) - grand)**2 for v in g.values()) / (len(g) - 1)
    msw = sum(((np.array(v) - np.mean(v))**2).sum() for v in g.values()) / (len(allv) - len(g))
    return (msb - msw) / (msb + (kbar - 1) * msw)

def cluster_se(A, B, idx):
    d = np.array([ov(A[k]) - ov(B[k]) for k in idx]); m = d.mean(); n = len(d)
    g = defaultdict(list)
    for k, x in zip(idx, d): g[k.split("|")[0]].append(x)
    G = len(g)
    num = sum((np.sum(np.array(v) - m))**2 for v in g.values())
    return m, d.std(ddof=1) / math.sqrt(n), math.sqrt(num / n**2 * (G / (G - 1))), G

PERSON = {"man","woman","men","women","person","people","boy","girl","boys","girls","guy","lady","businessman",
          "businesswoman","musician","musicians","child","children","kid","kids","baby","model","player","athlete",
          "worker","couple","gentleman","barber","tourist","tourists","doctor","doctors","assistant","assistants"}
_PREP = (r"\b(in|with|on|at|by|of|from|and|to|dressed|wearing|holding|using|reviewing|sitting|standing|"
         r"typing|sketching|working|reading|writing|playing|looking|smiling|drinking|reaching|leaning|posing|walking|browsing)\b")
def person_obj(instr):
    kinds = set()
    for p in re.split(r"(?<=\.)\s+", instr):
        if "<img>" not in p: continue
        m = re.search(r"(?:the\s+)?(.+?)\s+(?:is|are)\s+", p.lower())
        head = re.split(_PREP, m.group(1) if m else p.lower())[0].strip().split()
        kinds.add("person" if (head and head[-1] in PERSON) else "object")
    return "person" if kinds == {"person"} else ("object" if kinds == {"object"} else "mixed")

def strata(A, B, key, order, idx, PROV):
    print(f"  {key:12s} {'n':>4s} {'dPF':>8s} {'dSC':>8s} {'dOverall':>9s} {'p':>8s}")
    for lev in order:
        u = [k for k in idx if (PROV[k] if key == "provenance" else B[k][key]) == lev]
        if not u: continue
        dP = np.mean([A[k]["PF"] - B[k]["PF"] for k in u])
        dS = np.mean([A[k]["SC"] - B[k]["SC"] for k in u])
        d  = np.array([ov(A[k]) - ov(B[k]) for k in u])
        p  = stats.ttest_rel([ov(A[k]) for k in u], [ov(B[k]) for k in u])[1]
        print(f"  {str(lev):12s} {len(u):>4d} {dP:>+8.3f} {dS:>+8.3f} {d.mean():>+9.3f} {p:>8.3f}")

def agg(d, idx): return tuple(np.mean([f(d[k]) for k in idx]) for _, f in MET)
print(f"{'run':30s} {'PF':>7s} {'SC':>7s} {'Overall':>7s}")
print(f"{'prompting baseline (test)':30s} " + " ".join(f"{x:7.3f}" for x in agg(BASE, I)))
print(f"{'base, references removed':30s} " + " ".join(f"{x:7.3f}" for x in agg(ZERO, I)))
print(f"{'screen baseline':30s} " + " ".join(f"{x:7.3f}" for x in agg(SB, S)))
print(f"{'screen baseline, rerun':30s} " + " ".join(f"{x:7.3f}" for x in agg(SBR, S)))

PROV = {k: person_obj(META[k]["instruction"]) for k in I}
print("\nbaseline by stratum (test):")
for key, order in [("ref_kind", ["no_up","mixed","all_up"]), ("n_refs", [1,2,3]), ("provenance", ["person","object","mixed"])]:
    print(f"  {key:12s} " + " | ".join(
        f"{lev}: {np.mean([ov(BASE[k]) for k in I if (PROV[k] if key=='provenance' else BASE[k][key])==lev]):.3f} "
        f"(n={sum(1 for k in I if (PROV[k] if key=='provenance' else BASE[k][key])==lev)})" for lev in order))

run                                 PF      SC Overall
prompting baseline (test)        8.004   8.088   7.963
base, references removed         7.238   5.035   5.727
screen baseline                  8.017   8.079   7.955
screen baseline, rerun           8.093   8.106   8.010

baseline by stratum (test):
  ref_kind     no_up: 8.317 (n=157) | mixed: 8.205 (n=133) | all_up: 7.520 (n=198)
  n_refs       1: 7.994 (n=243) | 2: 7.958 (n=194) | 3: 7.832 (n=51)
  provenance   person: 8.557 (n=130) | object: 7.493 (n=224) | mixed: 8.173 (n=134)


## 3. Noise floor
Both sources are measured. `sd_reps` is the judge re-scoring one fixed set of images; the screen rerun is the whole pipeline repeated on an identical configuration in a later session. Generation noise, not judging, dominates.

In [3]:
reps, reps_sc = defaultdict(list), defaultdict(list)
for l in open(f"{SCORES}/sd_reps.jsonl", encoding="utf-8"):
    r = json.loads(l)
    if "error" not in r:
        reps[r["i"]].append(ov(r)); reps_sc[r["i"]].append(r["SC"])
sds = [np.std(v, ddof=1) for v in reps.values() if len(v) >= 2]
sigma_item = math.sqrt(np.mean(np.square(sds)))
print(f"judge re-scoring the same image : {len(reps)} items x {sorted({len(v) for v in reps.values()})} draws"
      f"  ->  per-item Overall SD {sigma_item:.3f}")

s = paired(SBR, SB, S, ov)
lo, hi = stats.chi2.interval(0.95, s["n"] - 1)
print(f"identical config, cross-session  : n={s['n']}  mean {s['mean']:+.3f} (p {s['p']:.3f})  sigma_d {s['sd']:.3f}"
      f"  95%CI [{s['sd']*math.sqrt((s['n']-1)/hi):.3f}, {s['sd']*math.sqrt((s['n']-1)/lo):.3f}]")
print(f"  judge-only floor sqrt(2)*sigma_item = {math.sqrt(2)*sigma_item:.3f}")
print(f"  implied generation-only sigma       = {math.sqrt(max(s['sd']**2/2 - sigma_item**2, 0)):.3f}")
print(f"  no mean shift detected in this identical-config rerun ({s['mean']:+.3f}, p {s['p']:.2f})")

# envelope: how much of the judge floor rests on a few dispersed items
_spread = lambda d: [max(v) - min(v) for v in d.values()]
print("\n  judge envelope:")
for _lab, _d in (("Overall", reps), ("SC", reps_sc)):
    print(f"    items with a within-item {_lab} spread >= 4: "
          f"{sum(1 for x in _spread(_d) if x >= 4)}/{len(_d)}  (widest {max(_spread(_d)):.2f})")
_keep = sorted(range(len(sds)), key=lambda j: sds[j])[:-3]
_si3 = math.sqrt(np.mean(np.square([sds[j] for j in _keep])))
print(f"    dropping the three most dispersed items: judge-only floor "
      f"{math.sqrt(2)*sigma_item:.3f} -> {math.sqrt(2)*_si3:.3f}, implied generation-only sigma "
      f"{math.sqrt(max(s['sd']**2/2 - sigma_item**2, 0)):.3f} -> "
      f"{math.sqrt(max(s['sd']**2/2 - _si3**2, 0)):.3f}")
print("\n  sigma_d by reference count:")
for k in (1, 2, 3):
    u = [x for x in S if SB[x]["n_refs"] == k]
    d = np.array([ov(SBR[x]) - ov(SB[x]) for x in u])
    print(f"    {k}-ref n={len(u):3d}  sigma_d {d.std(ddof=1):.3f}")
se488 = s["sd"] / math.sqrt(488)
print(f"\n  at n=488, at this floor, the SE of a paired mean delta is {se488:.3f} and the smallest")
print(f"  difference detectable at 80% power is {(1.96+0.8416)*se488:.3f} Overall")
print(f"  this is the floor, not the operating threshold: a lever that rewrites the prompt re-rolls")
print(f"  every row and carries a larger sigma_d. Section 4 prints the band that follows from it.")

judge re-scoring the same image : 90 items x [5] draws  ->  per-item Overall SD 0.362
identical config, cross-session  : n=302  mean +0.055 (p 0.327)  sigma_d 0.966  95%CI [0.895, 1.050]
  judge-only floor sqrt(2)*sigma_item = 0.513
  implied generation-only sigma       = 0.579
  no mean shift detected in this identical-config rerun (+0.055, p 0.33)

  judge envelope:
    items with a within-item Overall spread >= 4: 0/90  (widest 2.49)
    items with a within-item SC spread >= 4: 2/90  (widest 6.00)
    dropping the three most dispersed items: judge-only floor 0.513 -> 0.452, implied generation-only sigma 0.579 -> 0.604

  sigma_d by reference count:
    1-ref n=152  sigma_d 1.226
    2-ref n=120  sigma_d 0.591
    3-ref n= 30  sigma_d 0.643

  at n=488, at this floor, the SE of a paired mean delta is 0.044 and the smallest
  difference detectable at 80% power is 0.123 Overall
  this is the floor, not the operating threshold: a lever that rewrites the prompt re-rolls
  every row and c

## 4. Lever 1 — instruction rubrics
Five configurations against a screen baseline from the same session. `a5_filler` is neutral text of matched length: the contrast against it separates what the rubric *says* from the fact that something was prepended at all.

In [4]:
print("against the screen baseline:")
for a in L1: show(a, ARM[a], SB, S)
print("\nagainst a5_filler — the effect of rubric CONTENT (length held for a1/a2/a4; a3_both varies both content and length at 183 chars vs the filler's 101):")
for a in L1[:4]: show(f"{a} - a5_filler", ARM[a], ARM["a5_filler"], S, mets=[("Overall", ov)])

_sds = [paired(ARM[a], SB, S, ov)["sd"] for a in L1]
_arm = sum(_sds) / len(_sds)
_flr = paired(SBR, SB, S, ov)["sd"]
print(f"\nsigma_d across the five arms: {min(_sds):.3f}-{max(_sds):.3f}, mean {_arm:.3f}, "
      f"against {_flr:.3f} for an identical config")
print(f"  the band for an arm comparison follows from the arm sigma_d, not the floor:")
print(f"  at n=488, SE {_arm/math.sqrt(488):.3f} and MDE {(1.96+0.8416)*_arm/math.sqrt(488):.3f} Overall")
print(f"  the floor gives SE {_flr/math.sqrt(488):.3f} and MDE "
      f"{(1.96+0.8416)*_flr/math.sqrt(488):.3f}, which understates it")

against the screen baseline:
  a1_complete                      PF       n= 302  delta  +0.079  sigma_d 1.075  SE 0.062  p 0.2  90%CI [-0.023, +0.182]
  a1_complete                      SC       n= 302  delta  -0.020  sigma_d 1.494  SE 0.086  p 0.817  90%CI [-0.162, +0.122]
  a1_complete                      Overall  n= 302  delta  +0.027  sigma_d 1.227  SE 0.071  p 0.706  90%CI [-0.090, +0.143]
  a2_identity                      PF       n= 302  delta  -0.026  sigma_d 1.087  SE 0.063  p 0.672  90%CI [-0.130, +0.077]
  a2_identity                      SC       n= 302  delta  -0.043  sigma_d 1.447  SE 0.083  p 0.606  90%CI [-0.180, +0.094]
  a2_identity                      Overall  n= 302  delta  -0.037  sigma_d 1.217  SE 0.070  p 0.602  90%CI [-0.152, +0.079]
  a3_both                          PF       n= 302  delta  -0.050  sigma_d 1.051  SE 0.060  p 0.412  90%CI [-0.149, +0.050]
  a3_both                          SC       n= 302  delta  -0.096  sigma_d 1.663  SE 0.096  p 0.316  90%C

## 5. Lever 2 — negative prompt
The negative prompt is the only text channel the pipeline exposes besides the instruction. It enters `model_pred_ref` and `model_pred_uncond`, so it sets the anchor both guidance terms are measured from; `n_empty` removes it entirely.

In [5]:
for a in L2: show(a, ARM[a], SB, S)

  n_contrastive                    PF       n= 302  delta  -0.073  sigma_d 0.901  SE 0.052  p 0.161  90%CI [-0.158, +0.013]
  n_contrastive                    SC       n= 302  delta  -0.023  sigma_d 1.406  SE 0.081  p 0.775  90%CI [-0.157, +0.110]
  n_contrastive                    Overall  n= 302  delta  -0.039  sigma_d 1.043  SE 0.060  p 0.519  90%CI [-0.138, +0.060]
  n_contrastive2                   PF       n= 302  delta  -0.056  sigma_d 0.874  SE 0.050  p 0.264  90%CI [-0.139, +0.027]
  n_contrastive2                   SC       n= 302  delta  +0.040  sigma_d 1.262  SE 0.073  p 0.585  90%CI [-0.080, +0.160]
  n_contrastive2                   Overall  n= 302  delta  -0.018  sigma_d 1.016  SE 0.058  p 0.759  90%CI [-0.114, +0.078]
  n_filler                         PF       n= 302  delta  +0.017  sigma_d 0.990  SE 0.057  p 0.772  90%CI [-0.077, +0.111]
  n_filler                         SC       n= 302  delta  +0.007  sigma_d 1.447  SE 0.083  p 0.937  90%CI [-0.131, +0.144]
  n_fill

## 6. Excess per-row scatter
A lever can leave the mean untouched and still move individual rows. The identical-config pair gives the scatter attributable to the pipeline alone; each arm's delta is tested against it by a 20,000-draw Monte Carlo permutation on the variance ratio, which assumes within-pair exchangeability under the null but nothing about the shape of the delta distribution.

In [6]:
N_PERM = 20000
ref = {k: ov(SBR[k]) - ov(SB[k]) for k in S}

def perm_scatter(a, seed=0):
    cur = {k: ov(ARM[a][k]) - ov(SB[k]) for k in S}
    X = np.array([cur[k] for k in S]); Y = np.array([ref[k] for k in S]); m = len(S)
    ratio = X.var(ddof=1) / Y.var(ddof=1)
    sw = np.random.default_rng(seed).random((N_PERM, m)) < 0.5
    Xp, Yp = np.where(sw, Y, X), np.where(sw, X, Y)
    rp = Xp.var(axis=1, ddof=1) / Yp.var(axis=1, ddof=1)
    p = (1 + int((np.abs(np.log(rp)) >= abs(math.log(ratio))).sum())) / (N_PERM + 1)
    return X.std(ddof=1), ratio, p

print(f"  identical-config reference sigma_d = {np.std(list(ref.values()), ddof=1):.3f}")
print(f"  {'arm':18s} {'sigma_d':>8s} {'var ratio':>10s} {'perm p':>9s}")
for a in L1 + L2:
    sd, ratio, p = perm_scatter(a)
    print(f"  {a:18s} {sd:>8.3f} {ratio:>10.2f} {p:>9.4f}")

  identical-config reference sigma_d = 0.966
  arm                 sigma_d  var ratio    perm p
  a1_complete           1.227       1.61    0.0140
  a2_identity           1.217       1.59    0.1699
  a3_both               1.417       2.15    0.0132
  a4_style              1.338       1.92    0.0059
  a5_filler             1.315       1.85    0.0064
  n_contrastive         1.043       1.17    0.5406
  n_contrastive2        1.016       1.10    0.6918
  n_filler              1.066       1.22    0.4832
  n_empty               1.423       2.17    0.0011


## 7. TOST against the pre-registered ±0.15 bound
Equivalence is declared only if both one-sided tests reject, i.e. the 90% CI lies entirely inside the bound. A CI straddling the bound is reported as inconclusive, not as equivalent. The bound each arm could prove at 80% power depends on its own per-row spread and ranges from 0.171 to 0.240 on this slice, so ±0.15 sits below the floor for the noisier arms and the inconclusive verdicts are expected rather than surprising.

In [7]:
print(f"  {'arm':18s} {'delta':>8s} {'90% CI':>20s} {'TOST p':>8s}  verdict")
for a in L1 + L2:
    s, p_lo, p_hi, eq = tost(ARM[a], SB, S)
    ci = f"[{s['lo']:+.3f}, {s['hi']:+.3f}]"
    # the screen selected the configuration, so equivalence here is descriptive only;
    # confirmatory equivalence is claimed on the 488-row test split alone, as compare_runs enforces
    v  = "screen null, TOST passes descriptively" if eq else ("differs" if s["p"] < 0.05 else "inconclusive")
    print(f"  {a:18s} {s['mean']:>+8.3f} {ci:>20s} {max(p_lo, p_hi):>8.3f}  {v}")
b = sorted((1.645 + 1.2816) * paired(ARM[a], SB, S, ov)["se"] for a in L1 + L2)
print(f"\n  bound provable at 80% TOST power, per arm: {b[0]:.3f} to {b[-1]:.3f} "
      f"(the pre-registered 0.15 is below this range)")

  arm                   delta               90% CI   TOST p  verdict
  a1_complete          +0.027     [-0.090, +0.143]    0.041  screen null, TOST passes descriptively
  a2_identity          -0.037     [-0.152, +0.079]    0.053  inconclusive
  a3_both              -0.114     [-0.248, +0.021]    0.328  inconclusive
  a4_style             -0.115     [-0.242, +0.012]    0.325  inconclusive
  a5_filler            -0.010     [-0.135, +0.115]    0.033  screen null, TOST passes descriptively
  n_contrastive        -0.039     [-0.138, +0.060]    0.032  screen null, TOST passes descriptively
  n_contrastive2       -0.018     [-0.114, +0.078]    0.012  screen null, TOST passes descriptively
  n_filler             +0.029     [-0.072, +0.130]    0.024  screen null, TOST passes descriptively
  n_empty              -0.132     [-0.267, +0.003]    0.411  inconclusive

  bound provable at 80% TOST power, per arm: 0.171 to 0.240 (the pre-registered 0.15 is below this range)


## 8. Reference conditioning
The model-native in-context mechanism, and the only lever in arm A that moves the score. Measured on the full test split by removing the reference images.

In [8]:
show("references removed - baseline", ZERO, BASE, I)
print()
for key, order in [("ref_kind", ["no_up","mixed","all_up"]), ("n_refs", [1,2,3]), ("provenance", ["person","object","mixed"])]:
    strata(ZERO, BASE, key, order, I, PROV)
    print()

  references removed - baseline    PF       n= 488  delta  -0.766  sigma_d 1.969  SE 0.089  p 1.1e-16  90%CI [-0.913, -0.620]
  references removed - baseline    SC       n= 488  delta  -3.053  sigma_d 2.955  SE 0.134  p 5.92e-79  90%CI [-3.274, -2.833]
  references removed - baseline    Overall  n= 488  delta  -2.236  sigma_d 2.529  SE 0.114  p 3.69e-63  90%CI [-2.424, -2.047]

  ref_kind        n      dPF      dSC  dOverall        p
  no_up         157   -0.376   -3.389    -2.250    0.000
  mixed         133   -1.526   -3.045    -2.455    0.000
  all_up        198   -0.566   -2.793    -2.076    0.000

  n_refs          n      dPF      dSC  dOverall        p
  1             243   -0.399   -3.300    -2.350    0.000
  2             194   -1.077   -2.871    -2.149    0.000
  3              51   -1.333   -2.569    -2.023    0.000

  provenance      n      dPF      dSC  dOverall        p
  person        130   -0.231   -3.262    -2.129    0.000
  object        224   -0.652   -2.973    -2.201

## 9. Verdict
Lever 1 and lever 2 are read against the measured floor, not against zero.

In [9]:
best = max(L1, key=lambda a: abs(paired(ARM[a], SB, S, ov)["mean"]))
sb_ = paired(ARM[best], SB, S, ov)
print(f"  lever 1, largest absolute effect : {best} {sb_['mean']:+.3f} (p {sb_['p']:.3f})")
print(f"  lever 1, all five p-values       : " + ", ".join(f"{a.split('_')[0]} {paired(ARM[a], SB, S, ov)['p']:.3f}" for a in L1))
ne = paired(ARM["n_empty"], SB, S, ov)
print(f"  lever 2, only suggestive raw signal: n_empty {ne['mean']:+.3f} (p {ne['p']:.3f}), "
      f"SC {paired(ARM['n_empty'], SB, S, lambda r: r['SC'])['mean']:+.3f}")
zr = paired(ZERO, BASE, I, ov)
print(f"  lever 3, references removed      : {zr['mean']:+.3f} (p {zr['p']:.3g})")
print("\n  instruction wording is null on this pipeline: no configuration shifts the mean, while several")
print("  raise per-row scatter above the identical-config floor. Removing the negative prompt produced")
print("  a suggestive raw SC decrease on the screen, not established after multiplicity correction.")
print("  Reference conditioning is the only lever with a large effect, consistent with the architecture:")
print("  references reach the diffusion transformer through the VAE, never the MLLM text path.")

  lever 1, largest absolute effect : a4_style -0.115 (p 0.137)
  lever 1, all five p-values       : a1 0.706, a2 0.602, a3 0.165, a4 0.137, a5 0.894
  lever 2, only suggestive raw signal: n_empty -0.132 (p 0.109), SC -0.258
  lever 3, references removed      : -2.236 (p 3.69e-63)

  instruction wording is null on this pipeline: no configuration shifts the mean, while several
  raise per-row scatter above the identical-config floor. Removing the negative prompt produced
  a suggestive raw SC decrease on the screen, not established after multiplicity correction.
  Reference conditioning is the only lever with a large effect, consistent with the architecture:
  references reach the diffusion transformer through the VAE, never the MLLM text path.
